In [15]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm


import sys
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre


import torch.nn.functional as F
from pathlib import Path


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, TensorDataset
import time
import pickle
import networkx as nx
from sklearn.cluster import KMeans
import umap
from scipy.spatial import ConvexHull
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
# from function.Function import *

In [16]:
def count_array2_in_range_of_array1(array1, array2, threshold=5):

    sorted_array1 = np.sort(array1)
    array2 = np.sort(array2)
    
    lefts = array2 - threshold
    rights = array2 + threshold
    
    left_indices = np.searchsorted(sorted_array1, lefts, side='left')
    
    right_indices = np.searchsorted(sorted_array1, rights, side='right')
    
    has_within_range = right_indices > left_indices
    
    count = np.sum(has_within_range)
    
    return count

def label_array1_based_on_array2(array1, array2, threshold=5):
    array_1 = np.sort(array1)
    sorted_array2 = np.sort(array2)
    
    labels = np.zeros(len(array1), dtype=int)
    
    for i, value in enumerate(array1):
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        if right_index > left_index:
            labels[i] = 1
    
    return labels
def detect_local_minimum_in_window(data, window_size=20, std_multiplier=2):

    """
    在每个滑动窗口范围内检测局部最小值的索引，并确保最小值低于 mean - std_multiplier * std。

    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_rows, n_columns)。
    window_size : int
        滑动窗口的大小，用于定义局部范围，默认为 20。
    std_multiplier : float
        标准差的倍数，用于筛选局部最小值，默认为 2。

    返回:
    local_minima_indices : list of numpy.ndarray
        每行局部最小值的索引列表，每个元素是对应行局部最小值的索引数组。
    """
    local_minima_indices = []

    for row in data:
        minima_indices = []
        row = row.astype(np.float32)
        row_mean = np.mean(row)
        row_std = np.std(row)
        threshold = row_mean - std_multiplier * row_std

        for start in range(0, len(row), window_size):
            end = min(start + window_size, len(row))
            window = row[start:end]
            
            if len(window) > 0:
                local_min_index = np.argmin(window)
                local_min_value = window[local_min_index]
                
                if local_min_value < threshold:
                    minima_indices.append(start + local_min_index)  
        
        local_minima_indices.extend(minima_indices)
        local_minima_indices = list(set(local_minima_indices))  

    return local_minima_indices


def cluster_label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的 'time' 和 'cluster' 对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 的 'time' 中，则标记为对应的 'cluster' 值，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        包含 'time' 和 'cluster' 的二维数组。
        第一列为 'time'，第二列为 'cluster'。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 array2 中的 'cluster' 或 0。
    """

    array2 = np.array(array2.iloc[:, [5, 1]])
    sorted_indices = np.argsort(array2[:, 0])
    sorted_array2 = array2[sorted_indices]
    
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2[:, 0], left, side='left')
        right_index = np.searchsorted(sorted_array2[:, 0], right, side='right')
        
        # 如果范围内存在值，则标记为对应的 'cluster'
        if right_index > left_index:
            # 获取范围内的第一个匹配值的 'cluster'
            labels[i] = sorted_array2[left_index, 1]
    
    return labels


def label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的值对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 中，则标记为 1，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        用于判断的数组。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 0 或 1。
    """
    # 对 array2 进行排序以加速搜索
    sorted_array2 = np.sort(array2)
    
    # 初始化标签数组，默认值为 0
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        # 使用二分搜索判断范围内是否存在值
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        # 如果范围内存在值，则标记为 1
        if right_index > left_index:
            labels[i] = 1
    
    return labels


def extract_windows(data, indices, window_size=61):
    """
    根据给定的时间点索引提取窗口。
    
    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_channels, time)
    indices : numpy.ndarray
        时间点索引数组，用于指定需要提取窗口的中心点
    window_size : int
        窗口长度，默认为61（对应time-30到time+31）
    
    返回:
    windows : numpy.ndarray
        提取的窗口数据，形状为 (len(indices), n_channels, window_size)
    """
    n_channels, time_length = data.shape
    half_window = window_size // 2

    if np.any(indices < half_window) or np.any(indices >= time_length - half_window):
        raise ValueError("Some indices are out of bounds for the given window size.")

    windows = []
    for idx in indices:
        window = data[:, idx - half_window:idx + half_window + 1]
        windows.append(window)

    windows = np.array(windows)
    return windows

In [17]:
recording_raw = se.MEArecRecordingExtractor(file_path='/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_session1.h5')
probe_384channel = recording_raw.get_probegroup()
probe_384channel.set_global_device_channel_indices(range(384))
recording_raw = recording_raw.set_probegroup(probe_384channel)
recording_f = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")


In [18]:
def build_probe_group():
    print("[INFO] Loading probe template")
    template_recording = se.MEArecRecordingExtractor(file_path=str('/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_session1.h5'))
    probegroup = template_recording.get_probegroup()
    offset = 0
    for probe in probegroup.probes:
        n_contacts = probe.get_contact_count()
        device_indices = np.arange(offset, offset + n_contacts, dtype=int)
        probe.set_device_channel_indices(device_indices)
        offset += n_contacts
    return probegroup
from typing import Dict, Iterable, List, Sequence, Tuple

from dataclasses import dataclass
@dataclass
class CliqueInfo:
    clique_id: int
    device_channel_indices: List[int]
    contact_ids: List[str]
    center: Tuple[float, float]

In [19]:
def build_sliding_cliques(
    probe_group,
    clique_size: int = 50,
    min_size: int = 25,
    min_overlap: int = 16,
    target_groups: int = 11,
)-> List[CliqueInfo]:
    df = probe_group.to_dataframe()
    if "device_channel_indices" in df.columns:
        device_indices = df["device_channel_indices"].astype(int).to_numpy()
    else:
        device_indices = np.arange(len(df), dtype=int)
    positions = df.loc[:, ["x", "y"]].to_numpy()
    contact_ids = df["contact_ids"].astype(str).to_numpy()

    order = np.argsort(positions[:, 1])
    ordered_device = device_indices[order]
    ordered_contacts = contact_ids[order]
    ordered_positions = positions[order]

    step = clique_size - min_overlap
    cliques: List[CliqueInfo] = []

    start_indices = list(range(0, len(ordered_device) - clique_size + 1, step))
    if start_indices[-1] + clique_size < len(ordered_device):
        start_indices.append(len(ordered_device) - clique_size)

    for idx, start in enumerate(start_indices[:target_groups]):
        slice_device = ordered_device[start : start + clique_size]
        slice_positions = ordered_positions[start : start + clique_size]
        slice_contacts = ordered_contacts[start : start + clique_size]
        if len(slice_device) < min_size:
            continue
        center = tuple(np.mean(slice_positions, axis=0))
        cliques.append(
            CliqueInfo(
                clique_id=idx,
                device_channel_indices=list(slice_device),
                contact_ids=list(slice_contacts),
                center=center,
            )
        )

    print(f"[INFO] Built {len(cliques)} cliques (target {target_groups})")

    for clique in cliques:
        clique.device_channel_indices = [str(i + 1) for i in clique.device_channel_indices]
    return cliques

In [20]:
probe_group = build_probe_group()
cliques = build_sliding_cliques(
    probe_group,
    clique_size=50,
    
    min_size=25,
    min_overlap=16,
    target_groups=11,
)

[INFO] Loading probe template
[INFO] Built 11 cliques (target 11)


In [21]:
spike_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/session_1_spike_inf.csv")
cluster_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/session_1_cluster_inf.csv")

In [22]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    

class Spike_Detection_MLP(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, n_channels, time_window):
        super(Spike_Detection_MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, 16)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(16, output_size)
        self.sigmoid = nn.Sigmoid()  

        self.n_channels = n_channels
        self.time_window = time_window
    def forward(self, x):
        x = x.reshape(-1, self.n_channels * self.time_window)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        x = self.sigmoid(x)
        return x

In [23]:
probe_df = probe_group.to_dataframe()
probe_df['device_channel_indices'] = np.array([str(i + 1) for i in range(384)])

In [24]:
import json
from pathlib import Path
from typing import Dict, List, Tuple


def compute_clique_bounds(clique, probe_df: pd.DataFrame, cluster_info: pd.DataFrame, margin: float = 60.0) -> Tuple[np.ndarray, float, float]:
    """根据 clique 的空间范围筛选 cluster_info 中的神经元"""
    channel_mask = probe_df["device_channel_indices"].isin(clique.device_channel_indices)
    clique_positions = probe_df.loc[channel_mask, ["x", "y"]].to_numpy()

    if clique_positions.size == 0:
        y_min = float(cluster_info["y"].min())
        y_max = float(cluster_info["y"].max())
    else:
        if clique_positions.shape[0] >= 3:
            hull = ConvexHull(clique_positions)
            hull_points = clique_positions[hull.vertices]
        else:
            hull_points = clique_positions
        y_min = float(hull_points[:, 1].min())
        y_max = float(hull_points[:, 1].max())

    if y_max - y_min > 2 * margin:
        y_lower = y_min + margin
        y_upper = y_max - margin
    else:
        y_lower = y_min
        y_upper = y_max

    cluster_y = cluster_info["y"].to_numpy()
    cluster_ids = cluster_info["Neuron"].to_numpy()
    mask = (cluster_y >= y_lower) & (cluster_y <= y_upper)
    return cluster_ids[mask], y_lower, y_upper


def extract_clique_windows(
    recording: si.BaseRecording,
    clique,
    chunk_size: int,
    window_size: int,
    std_multiplier: float,
    start_frame: int = 0,
    end_frame = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """提取指定 clique 的候选窗口及索引"""
    total_samples = recording.get_num_samples()
    if end_frame is None:
        max_frame = total_samples
    else:
        max_frame = min(total_samples, end_frame)
    start_frame = max(0, start_frame)
    if start_frame >= max_frame:
        return np.empty((0, len(clique.device_channel_indices), window_size), dtype=np.float32), np.empty((0,), dtype=np.int64)

    half_window = window_size // 2
    all_valid_indices: List[int] = []
    all_windows: List[np.ndarray] = []

    for chunk_start in range(start_frame, max_frame, chunk_size):
        chunk_end = min(chunk_start + chunk_size, max_frame)
        data_chunk = recording.get_traces(
            start_frame=chunk_start,
            end_frame=chunk_end,
            channel_ids=clique.device_channel_indices,
        )

        threshold_result = detect_local_minimum_in_window(
            data_chunk.T,
            std_multiplier=std_multiplier,
            window_size=40,
        )

        if len(threshold_result) == 0:
            continue

        threshold_result = np.array(threshold_result) + chunk_start
        valid_indices = threshold_result[
            (threshold_result >= chunk_start + half_window + 1)
            & (threshold_result < chunk_end - half_window)
        ]

        if valid_indices.size == 0:
            continue

        rel_indices = valid_indices - chunk_start
        for rel_idx in rel_indices:
            window = data_chunk.T[:, rel_idx - half_window : rel_idx + half_window + 1]
            all_windows.append(window)
        all_valid_indices.extend(valid_indices)

    if not all_windows:
        return np.empty((0, len(clique.device_channel_indices), window_size), dtype=np.float32), np.empty((0,), dtype=np.int64)

    windows = np.stack(all_windows).astype(np.float32)
    indices = np.array(all_valid_indices, dtype=np.int64)
    return windows, indices


def assign_neuron_labels(
    spike_indices: np.ndarray,
    spike_windows: np.ndarray,
    spike_table: pd.DataFrame,
    threshold: int = 1,
) -> Tuple[np.ndarray, np.ndarray]:
    """基于 spike_table 的时间戳为窗口分配神经元标签"""
    if spike_windows.shape[0] == 0:
        return np.empty((0,), dtype=object), np.empty((0,), dtype=np.int64)

    sorted_times = spike_table["time"].to_numpy()
    sorted_neurons = spike_table["Neuron"].to_numpy()
    order = np.argsort(sorted_times)
    sorted_times = sorted_times[order]
    sorted_neurons = sorted_neurons[order]

    labels = np.empty(len(spike_indices), dtype=object)
    labels[:] = ""

    for i, idx in enumerate(spike_indices):
        left = idx - threshold
        right = idx + threshold
        left_index = np.searchsorted(sorted_times, left, side="left")
        right_index = np.searchsorted(sorted_times, right, side="right")
        if right_index > left_index:
            labels[i] = sorted_neurons[left_index]

    valid_mask = labels != ""
    return labels[valid_mask], np.where(valid_mask)[0]


def balance_by_neuron(
    windows: np.ndarray,
    neuron_labels: np.ndarray,
    max_samples_per_neuron: int = 8000,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, int]]:
    """对每个神经元做下采样并编码成整数标签"""
    encoded_labels: List[int] = []
    selected_indices: List[int] = []
    label_encoder: Dict[str, int] = {}

    unique_neurons = np.unique(neuron_labels)
    for idx, neuron in enumerate(unique_neurons):
        label_encoder[neuron] = idx
        neuron_indices = np.where(neuron_labels == neuron)[0]
        if len(neuron_indices) > max_samples_per_neuron:
            neuron_indices = np.random.choice(neuron_indices, max_samples_per_neuron, replace=False)
        selected_indices.extend(neuron_indices.tolist())
        encoded_labels.extend([idx] * len(neuron_indices))

    balanced_windows = windows[selected_indices]
    balanced_labels = np.array(encoded_labels, dtype=np.int64)
    shuffle_order = np.random.permutation(len(balanced_labels))
    return balanced_windows[shuffle_order], balanced_labels[shuffle_order], label_encoder


def save_label_encoder(path: Path, encoder: Dict[str, int]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(encoder, f, indent=2)


def load_label_encoder(path: Path) -> Dict[str, int]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def inverse_encoder(encoder: Dict[str, int]) -> Dict[int, str]:
    return {v: k for k, v in encoder.items()}


class ClassificationDataset(Dataset):
    def __init__(self, data: np.ndarray, labels: np.ndarray):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        return self.data[idx], self.labels[idx]


class SpikeClassificationMLP(nn.Module):
    def __init__(self, input_size: int, hidden_size1: int, hidden_size2: int, num_classes: int, proj_dim: int = 128):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.projection = nn.Sequential(
            nn.Linear(hidden_size2, hidden_size2),
            nn.ReLU(),
            nn.Linear(hidden_size2, proj_dim),
        )
        self.fc3 = nn.Linear(hidden_size2, num_classes)

    def forward(self, x: torch.Tensor, mode: str = "train"):
        x = x.reshape(x.shape[0], -1)
        x = self.relu1(self.fc1(x))
        features = self.relu2(self.fc2(x))
        if mode == "train":
            proj_features = self.projection(features)
            logits = self.fc3(features)
            return proj_features, logits, features
        if mode == "eval":
            return features
        raise ValueError(f"Unsupported mode: {mode}")



In [25]:
base_dir = Path("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/spike_detection/new")
classification_summary: List[Dict[str, float]] = []

chunk_size = 120000
window_size = 51
std_multiplier = 5.5
train_ratio = 0.8
batch_size = 1024
hidden_size1 = 256
hidden_size2 = 128
proj_dim = 128
learning_rate = 1e-4
max_epochs = 120
patience = 8
max_samples_per_neuron = 8000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] 分类训练将在 {device} 上运行")

for clique_id, clique in enumerate(cliques):
    print(f"\n[分类训练] 处理 clique {clique_id}")
    clique_dir = base_dir / str(clique_id)
    clique_dir.mkdir(parents=True, exist_ok=True)

    clique_neurons, _, _ = compute_clique_bounds(clique, probe_df, cluster_inf)
    if len(clique_neurons) == 0:
        print("  无匹配神经元，跳过")
        continue

    spike_inf_subset = spike_inf[spike_inf["Neuron"].isin(clique_neurons)].copy()
    if spike_inf_subset.empty:
        print("  对应的 spike 信息为空，跳过")
        continue

    windows, indices = extract_clique_windows(
        recording_f,
        clique,
        chunk_size=chunk_size,
        window_size=window_size,
        std_multiplier=std_multiplier,
    )

    if len(indices) == 0:
        print("  未检出候选事件，跳过")
        continue

    neuron_labels, valid_indices = assign_neuron_labels(indices, windows, spike_inf_subset, threshold=1)
    if len(neuron_labels) == 0:
        print("  候选事件未匹配到真实神经元，跳过")
        continue

    windows = windows[valid_indices]

    balanced_windows, balanced_labels, label_encoder = balance_by_neuron(
        windows,
        neuron_labels,
        max_samples_per_neuron=max_samples_per_neuron,
    )

    if len(balanced_labels) < 10:
        print("  有效样本过少，跳过")
        continue

    dataset = ClassificationDataset(balanced_windows, balanced_labels)
    train_size = int(train_ratio * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    input_size = balanced_windows.shape[1] * balanced_windows.shape[2]
    num_classes = len(label_encoder)

    global_best_acc = 0.0
    best_epoch = -1
    best_model_path = clique_dir / "classification_best.pth"
    encoder_path = clique_dir / "classification_label_encoder.json"

    for trail in range(1, 4):
        print(f"  Trail {trail} 开始训练 (classes={num_classes}, samples={len(dataset)})")
        model = SpikeClassificationMLP(input_size, hidden_size1, hidden_size2, num_classes, proj_dim=proj_dim)
        model = model.to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()

        best_trail_acc = 0.0
        patience_counter = 0

        for epoch in range(1, max_epochs + 1):
            model.train()
            train_correct = 0
            train_total = 0
            for batch_data, batch_labels in train_loader:
                batch_data = batch_data.to(device)
                batch_labels = batch_labels.to(device)

                _, logits, _ = model(batch_data, mode="train")
                loss = criterion(logits, batch_labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                preds = torch.argmax(logits, dim=1)
                train_correct += (preds == batch_labels).sum().item()
                train_total += batch_labels.size(0)

            train_acc = train_correct / max(train_total, 1)

            model.eval()
            val_correct = 0
            val_total = 0
            with torch.no_grad():
                for batch_data, batch_labels in val_loader:
                    batch_data = batch_data.to(device)
                    batch_labels = batch_labels.to(device)
                    _, logits, _ = model(batch_data, mode="train")
                    preds = torch.argmax(logits, dim=1)
                    val_correct += (preds == batch_labels).sum().item()
                    val_total += batch_labels.size(0)

            val_acc = val_correct / max(val_total, 1)
            if epoch % 5 == 0 or epoch == 1:
                print(f"    Epoch {epoch:03d} | Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f}")

            if val_acc > best_trail_acc + 1e-4:
                best_trail_acc = val_acc
                patience_counter = 0
                if val_acc > global_best_acc + 1e-4:
                    torch.save(model, best_model_path)
                    save_label_encoder(encoder_path, label_encoder)
                    global_best_acc = val_acc
                    best_epoch = epoch
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"    早停触发，最佳 Val Acc {best_trail_acc:.4f}")
                    break
        print(f"  Trail {trail} 完成，最佳 Val Acc {best_trail_acc:.4f}")

    if global_best_acc == 0:
        print("  未成功训练有效模型")
        continue

    best_model = torch.load(best_model_path, map_location=device, weights_only=False)
    best_model = best_model.to(device)
    best_model.eval()

    full_dataset = ClassificationDataset(balanced_windows, balanced_labels)
    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)

    feature_list = []
    with torch.no_grad():
        for batch_data, _ in full_loader:
            batch_data = batch_data.to(device)
            features = best_model(batch_data, mode="eval")
            feature_list.append(features.cpu().numpy())
    all_features = np.concatenate(feature_list, axis=0)

    feature_dim = all_features.shape[1]
    waveform_shape = (balanced_windows.shape[1], balanced_windows.shape[2])
    feature_means = np.zeros((num_classes, feature_dim), dtype=np.float32)
    waveform_means = np.zeros((num_classes, *waveform_shape), dtype=np.float32)
    neuron_ids = [None] * num_classes

    for neuron_id, idx in label_encoder.items():
        mask = balanced_labels == idx
        if not np.any(mask):
            continue
        feature_means[idx] = all_features[mask].mean(axis=0)
        waveform_means[idx] = balanced_windows[mask].mean(axis=0)
        neuron_ids[idx] = neuron_id

    prototypes_path = clique_dir / "classification_prototypes.npz"
    np.savez(
        prototypes_path,
        feature_means=feature_means,
        waveform_means=waveform_means,
        neuron_ids=np.array(neuron_ids, dtype=object),
    )

    classification_summary.append(
        {
            "clique_id": clique_id,
            "num_classes": num_classes,
            "samples": len(dataset),
            "best_val_acc": global_best_acc,
            "best_epoch": best_epoch,
        }
    )

classification_df = pd.DataFrame(classification_summary)
display(classification_df)
classification_metrics_path = base_dir / "classification_training_summary.csv"
classification_df.to_csv(classification_metrics_path, index=False)
print(f"\n[INFO] 分类训练统计已保存至 {classification_metrics_path}")



[INFO] 分类训练将在 cuda 上运行

[分类训练] 处理 clique 0
  Trail 1 开始训练 (classes=14, samples=65121)
    Epoch 001 | Train Acc 0.8100 | Val Acc 0.9694
    Epoch 005 | Train Acc 0.9974 | Val Acc 0.9869
    Epoch 010 | Train Acc 1.0000 | Val Acc 0.9888
    Epoch 015 | Train Acc 1.0000 | Val Acc 0.9889
    Epoch 020 | Train Acc 1.0000 | Val Acc 0.9893
    Epoch 025 | Train Acc 1.0000 | Val Acc 0.9894
    Epoch 030 | Train Acc 1.0000 | Val Acc 0.9895
    Epoch 035 | Train Acc 1.0000 | Val Acc 0.9893
    早停触发，最佳 Val Acc 0.9895
  Trail 1 完成，最佳 Val Acc 0.9895
  Trail 2 开始训练 (classes=14, samples=65121)
    Epoch 001 | Train Acc 0.8097 | Val Acc 0.9686
    Epoch 005 | Train Acc 0.9975 | Val Acc 0.9856
    Epoch 010 | Train Acc 1.0000 | Val Acc 0.9873
    Epoch 015 | Train Acc 1.0000 | Val Acc 0.9878
    Epoch 020 | Train Acc 1.0000 | Val Acc 0.9880
    Epoch 025 | Train Acc 1.0000 | Val Acc 0.9886
    Epoch 030 | Train Acc 1.0000 | Val Acc 0.9886
    早停触发，最佳 Val Acc 0.9888
  Trail 2 完成，最佳 Val Acc 0.9888
  Tra

,clique_id,num_classes,samples,best_val_acc,best_epoch
0,0,14,65121,0.989482,29
1,1,11,37922,0.991167,26
2,2,16,59782,0.984277,35
3,3,9,36802,0.992800,21
4,4,13,49320,0.985503,32
5,5,18,68021,0.985594,25
6,6,18,76189,0.983987,31
7,7,17,64119,0.989473,32
8,8,20,62438,0.979420,23
9,9,13,49204,0.987705,19



[INFO] 分类训练统计已保存至 /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/spike_detection/new/classification_training_summary.csv


In [29]:
session2_recording_path = base_dir.parent.parent / "data_generation" / "recording_neuropixels_session2.h5"
print(f"[INFO] Session2 数据路径: {session2_recording_path}")

recording_session2_raw = se.MEArecRecordingExtractor(file_path=str(session2_recording_path))
probe_session2 = recording_session2_raw.get_probegroup()
probe_session2.set_global_device_channel_indices(range(384))
recording_session2_raw = recording_session2_raw.set_probegroup(probe_session2)
recording_session2_f = spre.bandpass_filter(recording_session2_raw, freq_min=300, freq_max=3000)
recording_session2_f = spre.common_reference(recording_session2_f, reference="global", operator="median")

spike_inf_session2 = pd.read_csv(base_dir.parent.parent / "data_generation" / "session_2_spike_inf.csv")
cluster_inf_session2 = pd.read_csv(base_dir.parent.parent / "data_generation" / "session_2_cluster_inf.csv")

train_tpr_path = base_dir / "train_tpr_dict.pkl"
if train_tpr_path.exists():
    with open(train_tpr_path, "rb") as f:
        detection_tpr_dict = pickle.load(f)
else:
    detection_tpr_dict = {}

sampling_rate = recording_session2_f.get_sampling_frequency()
calibration_duration = 60  # seconds
calibration_frames = int(calibration_duration * sampling_rate)

session2_summary: List[Dict[str, float]] = []
all_predictions = []
all_mappings = []
all_feature_blocks: List[np.ndarray] = []
overall_correct = 0
overall_total = 0

for clique_id, clique in enumerate(cliques):
    print(f"\n[Session2 推理] clique {clique_id}")
    clique_dir = base_dir / str(clique_id)
    clique_dir.mkdir(parents=True, exist_ok=True)

    classification_path = clique_dir / "classification_best.pth"
    encoder_path = clique_dir / "classification_label_encoder.json"
    prototype_path = clique_dir / "classification_prototypes.npz"

    if not classification_path.exists() or not encoder_path.exists() or not prototype_path.exists():
        print("  分类权重或原型文件缺失，跳过")
        continue

    label_encoder = load_label_encoder(encoder_path)

    prototype_bundle = np.load(prototype_path, allow_pickle=True)
    feature_means = prototype_bundle["feature_means"]
    waveform_means = prototype_bundle["waveform_means"]
    neuron_ids = prototype_bundle["neuron_ids"].astype(object)

    valid_proto_mask = np.array([nid is not None for nid in neuron_ids])
    if not np.any(valid_proto_mask):
        print("  无有效原型，跳过")
        continue

    valid_feature_means = feature_means[valid_proto_mask]
    valid_neuron_ids = neuron_ids[valid_proto_mask]
    num_valid_classes = len(valid_neuron_ids)

    detection_scores = detection_tpr_dict.get(clique_id, [])
    if len(detection_scores) > 0:
        best_detection_trail = int(np.argmax(np.array(detection_scores))) + 1
    else:
        best_detection_trail = 1
    detection_model_path = clique_dir / f"trail_{best_detection_trail}.pth"
    if not detection_model_path.exists():
        available_detection = sorted(clique_dir.glob("trail_*.pth"))
        if not available_detection:
            print("  未找到检测模型，跳过")
            continue
        detection_model_path = available_detection[-1]
        best_detection_trail = int(detection_model_path.stem.split("_")[-1])

    detection_model = torch.load(detection_model_path, map_location=device, weights_only=False)
    detection_model = detection_model.to(device)
    detection_model.eval()

    classification_model = torch.load(classification_path, map_location=device, weights_only=False)
    classification_model = classification_model.to(device)
    classification_model.eval()

    clique_neurons, _, _ = compute_clique_bounds(clique, probe_df, cluster_inf_session2)
    spike_inf_subset = spike_inf_session2[spike_inf_session2["Neuron"].isin(clique_neurons)].copy()
    if spike_inf_subset.empty:
        print("  Clique 对应的真实神经元为空，跳过")
        continue

    # --- Calibration phase ---
    calib_windows, calib_indices = extract_clique_windows(
        recording_session2_f,
        clique,
        chunk_size=chunk_size,
        window_size=window_size,
        std_multiplier=std_multiplier,
        start_frame=0,
        end_frame=calibration_frames,
    )

    if len(calib_indices) == 0:
        print("  前60秒未检测到候选事件，跳过")
        continue

    detection_mask_list = []
    with torch.no_grad():
        for i in range(0, len(calib_windows), batch_size):
            batch = torch.tensor(calib_windows[i : i + batch_size], dtype=torch.float32, device=device)
            outputs = detection_model(batch).squeeze(1)
            detection_mask_list.append((outputs > 0.5).cpu().numpy())
    detection_mask = np.concatenate(detection_mask_list)
    calib_windows = calib_windows[detection_mask]
    calib_indices = calib_indices[detection_mask]

    if len(calib_indices) == 0:
        print("  检测模型未在前60秒识别到spike，跳过")
        continue

    calib_loader = DataLoader(TensorDataset(torch.tensor(calib_windows, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    calib_features = []
    with torch.no_grad():
        for (batch_data,) in calib_loader:
            batch_data = batch_data.to(device)
            features = classification_model(batch_data, mode="eval")
            calib_features.append(features.cpu().numpy())
    calib_features = np.concatenate(calib_features, axis=0)

    if calib_features.shape[0] == 0:
        print("  前60秒未获得特征，跳过")
        continue

    n_clusters = min(num_valid_classes, calib_features.shape[0])
    if n_clusters == 0:
        print("  无法建立聚类，跳过")
        continue

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    calib_cluster_labels = kmeans.fit_predict(calib_features)
    unique_clusters = np.unique(calib_cluster_labels)
    cluster_means = np.stack([calib_features[calib_cluster_labels == cid].mean(axis=0) for cid in unique_clusters])

    distance_matrix = cdist(cluster_means, valid_feature_means, metric="euclidean")
    row_ind, col_ind = linear_sum_assignment(distance_matrix)
    cluster_to_neuron = {}
    for r, c in zip(row_ind, col_ind):
        cluster_to_neuron[int(unique_clusters[r])] = str(valid_neuron_ids[c])

    for cid in unique_clusters:
        cid_int = int(cid)
        if cid_int in cluster_to_neuron:
            continue
        row_idx = np.where(unique_clusters == cid)[0][0]
        nearest_idx = int(np.argmin(distance_matrix[row_idx]))
        cluster_to_neuron[cid_int] = str(valid_neuron_ids[nearest_idx])

    calib_true_labels, calib_valid_positions = assign_neuron_labels(calib_indices, calib_windows, spike_inf_subset, threshold=1)
    if len(calib_true_labels) > 0:
        calib_pred_neurons = np.array([cluster_to_neuron[int(calib_cluster_labels[pos])] for pos in calib_valid_positions])
        calibration_accuracy = float(np.mean(calib_pred_neurons == calib_true_labels))
    else:
        calibration_accuracy = float("nan")

    mapping_path = clique_dir / "session2_cluster_mapping.json"
    with open(mapping_path, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in cluster_to_neuron.items()}, f, indent=2, ensure_ascii=False)

    kmeans_path = clique_dir / "session2_kmeans.pkl"
    with open(kmeans_path, "wb") as f:
        pickle.dump(kmeans, f)

    all_mappings.append(
        {
            "clique_id": clique_id,
            "num_clusters": int(len(unique_clusters)),
            "calibration_accuracy": calibration_accuracy,
        }
    )

    # --- Full recording inference ---
    full_windows, full_indices = extract_clique_windows(
        recording_session2_f,
        clique,
        chunk_size=chunk_size,
        window_size=window_size,
        std_multiplier=std_multiplier,
    )

    if len(full_indices) == 0:
        print("  全量数据未检测到候选事件，跳过")
        continue

    full_detection_mask_list = []
    with torch.no_grad():
        for i in range(0, len(full_windows), batch_size):
            batch = torch.tensor(full_windows[i : i + batch_size], dtype=torch.float32, device=device)
            outputs = detection_model(batch).squeeze(1)
            full_detection_mask_list.append((outputs > 0.5).cpu().numpy())
    full_detection_mask = np.concatenate(full_detection_mask_list)
    full_windows = full_windows[full_detection_mask]
    full_indices = full_indices[full_detection_mask]

    if len(full_indices) == 0:
        print("  检测模型未在全量数据中识别到spike，跳过")
        continue

    full_loader = DataLoader(TensorDataset(torch.tensor(full_windows, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    full_features = []
    with torch.no_grad():
        for (batch_data,) in full_loader:
            batch_data = batch_data.to(device)
            features = classification_model(batch_data, mode="eval")
            full_features.append(features.cpu().numpy())
    full_features = np.concatenate(full_features, axis=0)
    all_feature_blocks.append(full_features)

    cluster_full = kmeans.predict(full_features)
    predicted_neurons_full = np.array([cluster_to_neuron[int(cid)] for cid in cluster_full], dtype=object)

    true_labels_full, true_positions_full = assign_neuron_labels(full_indices, full_windows, spike_inf_subset, threshold=1)
    if len(true_labels_full) > 0:
        matched_predictions = predicted_neurons_full[true_positions_full]
        correct = int(np.sum(matched_predictions == true_labels_full))
        total = len(true_labels_full)
        accuracy = correct / total
    else:
        correct = 0
        total = 0
        accuracy = float("nan")

    overall_correct += correct
    overall_total += total

    prediction_df = pd.DataFrame(
        {
            "clique_id": clique_id,
            "time": full_indices,
            "cluster_id": cluster_full,
            "pred_neuron": predicted_neurons_full,
        }
    )
    prediction_df["true_neuron"] = np.nan
    prediction_df["correct"] = np.nan
    if len(true_labels_full) > 0:
        prediction_df.loc[true_positions_full, "true_neuron"] = true_labels_full
        prediction_df.loc[true_positions_full, "correct"] = prediction_df.loc[true_positions_full, "pred_neuron"] == prediction_df.loc[true_positions_full, "true_neuron"]

    prediction_path = clique_dir / f"session2_classification_predictions_clique_{clique_id}.csv"
    prediction_df.to_csv(prediction_path, index=False)

    embeddings_clique_path = clique_dir / f"session2_embeddings_clique_{clique_id}.npy"
    np.save(embeddings_clique_path, full_features)

    valid_mask_clique = prediction_df["true_neuron"].notnull() & prediction_df["pred_neuron"].notnull()
    if valid_mask_clique.any():
        valid_mask_array = valid_mask_clique.to_numpy()
        embeddings_valid = full_features[valid_mask_array]
        true_labels_clique = prediction_df.loc[valid_mask_clique, "true_neuron"].to_numpy()
        pred_labels_clique = prediction_df.loc[valid_mask_clique, "pred_neuron"].to_numpy()

        if embeddings_valid.shape[0] > 1:
            sample_size = min(50000, embeddings_valid.shape[0])
            rng = np.random.default_rng(42)
            if sample_size < embeddings_valid.shape[0]:
                sample_indices = rng.choice(embeddings_valid.shape[0], size=sample_size, replace=False)
            else:
                sample_indices = np.arange(embeddings_valid.shape[0])

            umap_model_clique = umap.UMAP(n_components=2, random_state=42)
            embeddings_2d_clique = umap_model_clique.fit_transform(embeddings_valid[sample_indices])

            umap_pdf_clique = clique_dir / f"session2_umap_clique_{clique_id}.pdf"
            with PdfPages(umap_pdf_clique) as pdf:
                plt.figure(figsize=(8, 8))
                sns.scatterplot(
                    x=embeddings_2d_clique[:, 0],
                    y=embeddings_2d_clique[:, 1],
                    hue=true_labels_clique[sample_indices],
                    alpha=0.8,
                    s=2,
                    legend=False,
                )
                plt.title("UMAP Visualization - True Labels")
                plt.grid(False)
                pdf.savefig()
                plt.close()

                plt.figure(figsize=(8, 8))
                sns.scatterplot(
                    x=embeddings_2d_clique[:, 0],
                    y=embeddings_2d_clique[:, 1],
                    hue=pred_labels_clique[sample_indices],
                    alpha=0.8,
                    s=2,
                    legend=False,
                )
                plt.title("UMAP Visualization - Predicted Labels")
                plt.grid(False)
                pdf.savefig()
                plt.close()

        confusion_counts_clique = pd.crosstab(
            prediction_df.loc[valid_mask_clique, "true_neuron"],
            prediction_df.loc[valid_mask_clique, "pred_neuron"],
            dropna=False,
        )
        confusion_counts_clique_path = clique_dir / f"session2_confusion_counts_clique_{clique_id}.csv"
        confusion_counts_clique.to_csv(confusion_counts_clique_path)

        confusion_normalized_clique = confusion_counts_clique.div(
            confusion_counts_clique.sum(axis=1).replace(0, np.nan), axis=0
        ).fillna(0)
        confusion_normalized_clique_path = clique_dir / f"session2_confusion_normalized_clique_{clique_id}.csv"
        confusion_normalized_clique.to_csv(confusion_normalized_clique_path)

        heatmap_pdf_clique = clique_dir / f"session2_confusion_heatmap_clique_{clique_id}.pdf"
        with PdfPages(heatmap_pdf_clique) as pdf:
            plt.figure(figsize=(12, 10))
            sns.heatmap(confusion_normalized_clique, cmap="Blues")
            plt.title("Normalized Confusion Matrix (True vs Predicted)")
            pdf.savefig()
            plt.close()

    session2_summary.append(
        {
            "clique_id": clique_id,
            "samples": int(len(full_indices)),
            "matched_samples": int(total),
            "accuracy": accuracy,
            "calibration_accuracy": calibration_accuracy,
            "num_clusters": int(len(unique_clusters)),
        }
    )
    all_predictions.append(prediction_df)

if session2_summary:
    session2_df = pd.DataFrame(session2_summary)
    display(session2_df)
    overall_accuracy = overall_correct / overall_total if overall_total > 0 else float("nan")
    print(f"\n[INFO] Session2 总体准确率: {overall_accuracy:.4f} ({overall_correct}/{overall_total})")
    session2_summary_path = base_dir / "session2_classification_summary.csv"
    session2_df.to_csv(session2_summary_path, index=False)
    print(f"[INFO] Session2 分类汇总已保存至 {session2_summary_path}")

    mapping_df = pd.DataFrame(all_mappings)
    mapping_path = base_dir / "session2_cluster_mapping_summary.csv"
    mapping_df.to_csv(mapping_path, index=False)
    print(f"[INFO] KMeans 映射信息已保存至 {mapping_path}")

    predictions_concat = pd.concat(all_predictions, ignore_index=True)
    predictions_path = base_dir / "session2_classification_predictions_all.csv"
    predictions_concat.to_csv(predictions_path, index=False)
    print(f"[INFO] Session2 详细预测已保存至 {predictions_path}")

    final_spike_inf_unsorted = predictions_concat.rename(
        columns={
            "cluster_id": "cluster_predicted",
            "pred_neuron": "neuron_id",
        }
    )
    final_spike_inf_unsorted = final_spike_inf_unsorted[["time", "clique_id", "cluster_predicted", "neuron_id", "true_neuron", "correct"]]
    final_spike_inf_unsorted = final_spike_inf_unsorted.reset_index(drop=True)

    if all_feature_blocks:
        all_features = np.concatenate(all_feature_blocks, axis=0)
        embeddings_path = base_dir / "session2_final_embeddings_overall.npy"
        np.save(embeddings_path, all_features)
        print(f"[INFO] final embeddings 已保存至 {embeddings_path}")
    else:
        all_features = np.empty((0, 0), dtype=np.float32)

    final_spike_inf = final_spike_inf_unsorted.sort_values("time").reset_index(drop=True)
    final_spike_inf_path = base_dir / "session2_final_spike_inf.csv"
    final_spike_inf.to_csv(final_spike_inf_path, index=False)
    print(f"[INFO] final_spike_inf 已保存至 {final_spike_inf_path}")

    valid_mask = final_spike_inf_unsorted["neuron_id"].notnull() & final_spike_inf_unsorted["true_neuron"].notnull()
    if all_features.size > 0 and valid_mask.any():
        valid_embeddings = all_features[valid_mask.to_numpy()]
        valid_labels_true = final_spike_inf_unsorted.loc[valid_mask, "true_neuron"].to_numpy()
        valid_labels_pred = final_spike_inf_unsorted.loc[valid_mask, "neuron_id"].to_numpy()

        sample_size = min(100000, valid_embeddings.shape[0])
        rng = np.random.default_rng(42)
        if sample_size < valid_embeddings.shape[0]:
            sample_indices = rng.choice(valid_embeddings.shape[0], size=sample_size, replace=False)
        else:
            sample_indices = np.arange(valid_embeddings.shape[0])

        umap_model = umap.UMAP(n_components=2, random_state=42)
        embeddings_2d = umap_model.fit_transform(valid_embeddings[sample_indices])

        umap_pdf_path = base_dir / "session2_umap_overall.pdf"
        with PdfPages(umap_pdf_path) as pdf:
            plt.figure(figsize=(8, 8))
            sns.scatterplot(
                x=embeddings_2d[:, 0],
                y=embeddings_2d[:, 1],
                hue=valid_labels_true[sample_indices],
                alpha=0.8,
                s=2,
                legend=False,
            )
            plt.title("UMAP Visualization - True Labels")
            plt.grid(False)
            pdf.savefig()
            plt.close()

            plt.figure(figsize=(8, 8))
            sns.scatterplot(
                x=embeddings_2d[:, 0],
                y=embeddings_2d[:, 1],
                hue=valid_labels_pred[sample_indices],
                alpha=0.8,
                s=2,
                legend=False,
            )
            plt.title("UMAP Visualization - Predicted Labels")
            plt.grid(False)
            pdf.savefig()
            plt.close()
        print(f"[INFO] UMAP 可视化已保存至 {umap_pdf_path}")

        confusion_counts = pd.crosstab(
            final_spike_inf_unsorted.loc[valid_mask, "true_neuron"],
            final_spike_inf_unsorted.loc[valid_mask, "neuron_id"],
            dropna=False,
        )
        confusion_counts_path = base_dir / "session2_confusion_counts_overall.csv"
        confusion_counts.to_csv(confusion_counts_path)
        print(f"[INFO] 混淆矩阵计数已保存至 {confusion_counts_path}")

        confusion_normalized = confusion_counts.div(confusion_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
        confusion_normalized_path = base_dir / "session2_confusion_normalized_overall.csv"
        confusion_normalized.to_csv(confusion_normalized_path)
        print(f"[INFO] 混淆矩阵归一化结果已保存至 {confusion_normalized_path}")

        heatmap_pdf_path = base_dir / "session2_confusion_heatmap_overall.pdf"
        with PdfPages(heatmap_pdf_path) as pdf:
            plt.figure(figsize=(12, 10))
            sns.heatmap(confusion_normalized, cmap="Blues")
            plt.title("Normalized Confusion Matrix (True vs Predicted)")
            pdf.savefig()
            plt.close()
        print(f"[INFO] 混淆矩阵热图已保存至 {heatmap_pdf_path}")

else:
    print("[WARN] 未生成任何 Session2 分类结果")



[INFO] Session2 数据路径: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_session2.h5

[Session2 推理] clique 0

[Session2 推理] clique 1

[Session2 推理] clique 2

[Session2 推理] clique 3

[Session2 推理] clique 4

[Session2 推理] clique 5

[Session2 推理] clique 6

[Session2 推理] clique 7

[Session2 推理] clique 8

[Session2 推理] clique 9


KeyboardInterrupt: 